In [2]:
# Gaussian-process regression for foam property prediction
# Train one GPR model per response, evaluate leave-one-out performance,
# and compare predictions against the measured formulation data.
# Generate 100,000 continuous candidate formulas and predict all 3 properties.

In [3]:
# import data

from pathlib import Path
import pandas as pd

# Load the compiled data
project_root = Path.cwd().resolve().parent if 'notebooks' in str(Path.cwd()) else Path.cwd().resolve()
excel_path = project_root / 'data' / 'raw' / 'Compiled Data.xlsx'

df = pd.read_excel(excel_path)

print()

In [4]:
# Show the 25 original formulas and their three measured targets from df.
formula_cols = ["TEM", "Joncryl", "NC513", "PEG-DE", "NGDE"]
measured_target_cols = [
    "Avg_Density_g_cm3",
    "Dry Strength Mean",
    "Wet Strength Mean",
]
raw_data_summary = df[formula_cols + measured_target_cols].copy()
raw_data_summary[formula_cols] = raw_data_summary[formula_cols].fillna(0.0)
raw_data_summary[measured_target_cols] = raw_data_summary[measured_target_cols].apply(
    pd.to_numeric, errors="raise"
)

assert len(raw_data_summary) == 25, (
    f"Expected the 25 original samples; found {len(raw_data_summary)}."
)
display(raw_data_summary)

,TEM,Joncryl,NC513,PEG-DE,NGDE,Avg_Density_g_cm3,Dry Strength Mean,Wet Strength Mean
0,8,0.0,0.0,0.0,0.0,0.192000,53.674733,23.680000
1,8,0.7,0.0,0.0,0.0,0.195000,68.863717,24.258481
2,8,1.0,0.0,0.0,0.0,0.159000,50.405845,21.329450
3,8,1.5,0.0,0.0,0.0,0.159933,54.717605,20.183592
4,8,2.0,0.0,0.0,0.0,0.201000,85.787785,27.585357
5,8,3.0,0.0,0.0,0.0,0.202000,80.535443,27.921901
6,5,0.0,0.0,0.0,0.0,0.298930,157.659876,51.133703
7,5,0.0,0.7,0.0,0.0,0.360035,215.989325,82.812461
8,5,0.0,1.0,0.0,0.0,0.355557,218.543779,77.134354
9,5,0.0,2.0,0.0,0.0,0.358055,196.341194,76.406460


In [ ]:
# Train one Gaussian process regressor per target, comparing several kernel families
# with nested LOOCV, then fitting the selected model on all 25 observations.

import time
import numpy as np
import pandas as pd
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    RationalQuadratic,
    RBF,
    WhiteKernel,
)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import LeaveOneOut
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

feature_cols = ["TEM", "Joncryl", "NC513", "PEG-DE", "NGDE"]
target_cols = [
    "Avg_Density_g_cm3",
    "Dry Strength Mean",
    "Wet Strength Mean",
]

n_restarts_optimizer = 0
random_seed = 42
time_budget_seconds = 600
search_started_at = time.monotonic()


def check_time_budget():
    elapsed = time.monotonic() - search_started_at
    if elapsed >= time_budget_seconds:
        raise TimeoutError(
            f"Nested GP search exceeded its {time_budget_seconds // 60}-minute budget "
            f"after {elapsed / 60:.1f} minutes. No partial results or models were saved."
        )


def fit_with_budget(estimator, X_train, y_train):
    check_time_budget()
    estimator.fit(X_train, y_train)
    check_time_budget()
    return estimator


def standardized_rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    scale = np.std(y_true, ddof=1)
    return np.sqrt(mean_squared_error(y_true, y_pred)) / scale if scale > 0 else np.sqrt(
        mean_squared_error(y_true, y_pred)
    )


def build_gp_estimator(kernel):
    return make_pipeline(
        StandardScaler(),
        GaussianProcessRegressor(
            kernel=kernel,
            normalize_y=True,
            n_restarts_optimizer=n_restarts_optimizer,
            random_state=random_seed,
        ),
    )


def make_kernel(kernel_name, n_features):
    length_scale = np.ones(n_features)
    noise_term = WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-8, 1e1))
    constant_term = ConstantKernel(1.0, (1e-3, 1e3))

    if kernel_name == "RBF":
        signal = RBF(length_scale=length_scale, length_scale_bounds=(1e-2, 1e3))
    elif kernel_name == "Matern_1.5":
        signal = Matern(
            length_scale=length_scale,
            length_scale_bounds=(1e-2, 1e3),
            nu=1.5,
        )
    elif kernel_name == "Matern_2.5":
        signal = Matern(
            length_scale=length_scale,
            length_scale_bounds=(1e-2, 1e3),
            nu=2.5,
        )
    elif kernel_name == "RQ":
        # sklearn's RationalQuadratic supports a shared scalar length scale, not ARD.
        signal = RationalQuadratic(
            length_scale=1.0,
            length_scale_bounds=(1e-2, 1e3),
            alpha=1.0,
            alpha_bounds=(1e-3, 1e3),
        )
    else:
        raise ValueError(f"Unsupported kernel name: {kernel_name}")

    return constant_term * signal + noise_term


kernel_names = ["RBF", "Matern_1.5", "Matern_2.5", "RQ"]


def select_kernel_name(X_train, y_train):
    scores = {}
    inner_loo = LeaveOneOut()
    y_array = np.asarray(y_train, dtype=float)

    for kernel_name in kernel_names:
        fold_scores = []
        for inner_train_idx, inner_val_idx in inner_loo.split(X_train):
            inner_y_train = y_array[inner_train_idx]
            estimator = build_gp_estimator(make_kernel(kernel_name, X_train.shape[1]))
            fit_with_budget(
                estimator,
                X_train.iloc[inner_train_idx],
                inner_y_train,
            )
            prediction = estimator.predict(X_train.iloc[inner_val_idx])
            scale = np.std(inner_y_train, ddof=1)
            if scale <= 0:
                scale = 1.0
            fold_scores.append(
                np.sqrt(
                    mean_squared_error(y_array[inner_val_idx], prediction)
                ) / scale
            )
        scores[kernel_name] = float(np.mean(fold_scores))

    return min(scores, key=scores.get), scores


# Blank ingredient cells mean that ingredient was not used in the formulation.
model_data = df[feature_cols + target_cols].copy()
model_data[feature_cols] = model_data[feature_cols].fillna(0.0)
model_data[target_cols] = model_data[target_cols].apply(pd.to_numeric, errors="raise")
assert len(model_data) == 25, f"Expected all 25 samples; found {len(model_data)}."
assert not model_data[target_cols].isna().any().any(), "A target contains missing values."

X = model_data[feature_cols]
outer_loo = LeaveOneOut()
gpr_models = {}
cv_predictions = pd.DataFrame(index=model_data.index)
results = []

for target in target_cols:
    y = model_data[target]
    outer_predictions = np.empty(len(X), dtype=float)
    outer_selected_kernels = []

    for train_idx, test_idx in outer_loo.split(X):
        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]
        selected_kernel, _ = select_kernel_name(X_train, y_train)
        outer_selected_kernels.append(selected_kernel)
        estimator = build_gp_estimator(make_kernel(selected_kernel, X_train.shape[1]))
        fit_with_budget(estimator, X_train, y_train)
        outer_predictions[test_idx] = estimator.predict(X.iloc[test_idx])

    cv_predictions[f"{target} - nested LOOCV prediction"] = outer_predictions

    # Select the final kernel by LOOCV on all available training data, then refit.
    final_kernel_name, final_kernel_scores = select_kernel_name(X, y)
    final_model = build_gp_estimator(make_kernel(final_kernel_name, X.shape[1]))
    gpr_models[target] = fit_with_budget(final_model, X, y)

    results.append(
        {
            "target": target,
            "final selected kernel": final_kernel_name,
            "nested LOOCV R2": r2_score(y, outer_predictions),
            "nested LOOCV MAE": mean_absolute_error(y, outer_predictions),
            "nested LOOCV RMSE": np.sqrt(mean_squared_error(y, outer_predictions)),
            "nested LOOCV standardized RMSE": standardized_rmse(y, outer_predictions),
            "kernel selection counts in outer folds": str(
                pd.Series(outer_selected_kernels).value_counts().to_dict()
            ),
            "final kernel LOOCV scores": str(final_kernel_scores),
            "optimized kernel (all 25 samples)": str(
                gpr_models[target].named_steps["gaussianprocessregressor"].kernel_
            ),
        }
    )

results_df = pd.DataFrame(results).set_index("target")
print("Nested-LOOCV GP tuning results:")
display(results_df)
print(f"Search runtime: {(time.monotonic() - search_started_at) / 60:.1f} minutes")
print("\nFirst five out-of-fold predictions:")
display(cv_predictions.head())



In [ ]:
# save the 3 models to the folder models

from pathlib import Path
import joblib

project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
models_dir = project_root / "models"
models_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(gpr_models, models_dir / "gpr_models.joblib")

In [ ]:
# Compare leave-one-out predictions with the measured values for each target.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(target_cols), figsize=(16, 5))

for ax, target in zip(axes, target_cols):
    y_true = model_data[target]
    y_pred = cv_predictions[f"{target} - LOOCV prediction"]
    score = r2_score(y_true, y_pred)
    lower = min(y_true.min(), y_pred.min())
    upper = max(y_true.max(), y_pred.max())
    padding = (upper - lower) * 0.05
    plot_min, plot_max = lower - padding, upper + padding

    ax.scatter(y_true, y_pred, alpha=0.8, edgecolor="black", linewidth=0.5)
    ax.plot([plot_min, plot_max], [plot_min, plot_max], "--", color="gray", label="Ideal prediction")
    ax.set_xlim(plot_min, plot_max)
    ax.set_ylim(plot_min, plot_max)
    ax.set_aspect("equal", adjustable="box")
    ax.set_title(f"{target}\nLOOCV $R^2$ = {score:.3f}")
    ax.set_xlabel("True value")
    ax.set_ylabel("LOOCV predicted value")
    ax.grid(alpha=0.25)
    ax.legend()

fig.tight_layout()
plt.show()

In [ ]:
# Show the distribution of absolute errors across the 25 LOOCV folds.

absolute_errors = pd.DataFrame(index=model_data.index)
for target in target_cols:
    absolute_errors[target] = (
        model_data[target] - cv_predictions[f"{target} - LOOCV prediction"]
    ).abs()

fig, axes = plt.subplots(1, len(target_cols), figsize=(16, 5))

for ax, target in zip(axes, target_cols):
    errors = absolute_errors[target]
    ax.hist(errors, bins="auto", edgecolor="black", alpha=0.75)
    ax.axvline(
        errors.mean(),
        color="red",
        linestyle="--",
        label=f"Mean absolute error = {errors.mean():.3g}",
    )
    ax.set_title(f"{target}\nLOOCV absolute errors (n={len(errors)})")
    ax.set_xlabel("Absolute error")
    ax.set_ylabel("Number of held-out samples")
    ax.grid(axis="y", alpha=0.25)
    ax.legend()

fig.tight_layout()
plt.show()

In [ ]:
# Plot predictions from the final GPR estimators fitted on all 25 samples.
import matplotlib.pyplot as plt

training_predictions = pd.DataFrame(index=model_data.index)
for target in target_cols:
    training_predictions[target] = gpr_models[target].predict(X)

fig, axes = plt.subplots(1, len(target_cols), figsize=(16, 5))

for ax, target in zip(axes, target_cols):
    y_true = model_data[target]
    y_pred = training_predictions[target]
    training_r2 = r2_score(y_true, y_pred)
    lower = min(y_true.min(), y_pred.min())
    upper = max(y_true.max(), y_pred.max())
    padding = (upper - lower) * 0.05
    plot_min, plot_max = lower - padding, upper + padding

    ax.scatter(y_true, y_pred, alpha=0.8, edgecolor="black", linewidth=0.5)
    ax.plot(
        [plot_min, plot_max],
        [plot_min, plot_max],
        "--",
        color="gray",
        label="Ideal prediction",
    )
    ax.set_xlim(plot_min, plot_max)
    ax.set_ylim(plot_min, plot_max)
    ax.set_aspect("equal", adjustable="box")
    ax.set_title(f"{target}\nTraining $R^2$ = {training_r2:.3f}")
    ax.set_xlabel("Actual value")
    ax.set_ylabel("Final model prediction")
    ax.grid(alpha=0.25)
    ax.legend()

fig.tight_layout()
plt.show()

In [ ]:
# Generate continuous candidate formulas within the feasible ranges.
# - TEM: 5-8% with no grid rounding.
# - Each chain extender (Joncryl, NC513, PEG-DE, NGDE): 0.7-3% when included; 0% when omitted.
# - Combined chain extender amount: no more than 6%.
# - Generate 100,000 valid candidate formulas.
import numpy as np

candidate_count = 100_000
chain_extender_cols = ["Joncryl", "NC513", "PEG-DE", "NGDE"]
feature_cols = ["TEM", *chain_extender_cols]
rng = np.random.default_rng(42)

candidate_chunks = []
collected = 0
while collected < candidate_count:
    batch_size = max(10_000, 2 * (candidate_count - collected))
    active_counts = rng.integers(1, len(chain_extender_cols) + 1, size=batch_size)
    active_order = rng.random((batch_size, len(chain_extender_cols))).argsort(axis=1)
    active_mask = active_order < active_counts[:, None]
    extender_amounts = rng.uniform(
        0.7, 3.0, size=(batch_size, len(chain_extender_cols))
    )
    extender_amounts = np.where(active_mask, extender_amounts, 0.0)

    valid_mask = extender_amounts.sum(axis=1) <= 6.0
    valid_batch = extender_amounts[valid_mask]
    candidate_chunks.append(valid_batch)
    collected += len(valid_batch)

extender_values = np.concatenate(candidate_chunks, axis=0)[:candidate_count]
candidate_set = pd.DataFrame(extender_values, columns=chain_extender_cols)
candidate_set.insert(0, "TEM", rng.uniform(5.0, 8.0, size=candidate_count))
candidate_set = candidate_set[feature_cols]

# Confirm the generated formulas satisfy every requested bound.
assert len(candidate_set) == candidate_count
assert candidate_set["TEM"].between(5.0, 8.0).all()
assert candidate_set[chain_extender_cols].sum(axis=1).le(6.0).all()
for col in chain_extender_cols:
    active = candidate_set[col].ne(0)
    assert candidate_set.loc[active, col].between(0.7, 3.0).all()

print(f"Generated {len(candidate_set):,} valid continuous candidate formulas.")
print(candidate_set.head())
print("Maximum total chain extender:", candidate_set[chain_extender_cols].sum(axis=1).max())

In [ ]:
# Predict the mean and predictive standard deviation for each generated formula.
candidate_predictions = candidate_set.copy()

for target in target_cols:
    mean, std = gpr_models[target].predict(
        candidate_set[feature_cols],
        return_std=True,
    )
    candidate_predictions[f"predicted_{target}"] = mean
    candidate_predictions[f"std_{target}"] = std

prediction_cols = [f"predicted_{target}" for target in target_cols]
std_cols = [f"std_{target}" for target in target_cols]
assert len(candidate_predictions) == candidate_count
assert all(col in candidate_predictions.columns for col in prediction_cols + std_cols)
assert np.isfinite(candidate_predictions[prediction_cols + std_cols].to_numpy()).all()
assert candidate_predictions[std_cols].ge(0).all().all()

print(f"Predictions and predictive standard deviations generated for {len(candidate_predictions):,} candidates.")
display(candidate_predictions.head())

In [ ]:
# Keep candidates that are not dominated on all three predicted properties.
# Objectives: minimize density; maximize dry and wet strength.
density_col = "predicted_Avg_Density_g_cm3"
dry_strength_col = "predicted_Dry Strength Mean"
wet_strength_col = "predicted_Wet Strength Mean"
objective_cols = [density_col, dry_strength_col, wet_strength_col]
std_cols = [f"std_{target}" for target in target_cols]
objective_values = candidate_predictions[objective_cols].to_numpy(dtype=float)
finite_mask = np.isfinite(objective_values).all(axis=1)
finite_indices = np.flatnonzero(finite_mask)
finite_values = objective_values[finite_mask]

# Keep one representative if multiple formulas have identical predicted objectives.
unique_values, first_positions = np.unique(finite_values, axis=0, return_index=True)
original_indices = finite_indices[first_positions]

# Sort by density ascending, then dry and wet strength descending.
order = np.lexsort((-unique_values[:, 2], -unique_values[:, 1], unique_values[:, 0]))
sorted_values = unique_values[order]
sorted_indices = original_indices[order]
dry_strength_levels = np.unique(sorted_values[:, 1])
level_count = len(dry_strength_levels)
fenwick_max_wet_strength = np.full(level_count + 1, -np.inf)
pareto_indices = []

for row, original_index in zip(sorted_values, sorted_indices):
    density, dry_strength, wet_strength = row
    dry_strength_rank = np.searchsorted(dry_strength_levels, dry_strength)
    query_index = level_count - dry_strength_rank

    # Query the best wet strength among earlier candidates with at least this dry strength.
    best_prior_wet_strength = -np.inf
    index = query_index
    while index > 0:
        best_prior_wet_strength = max(
            best_prior_wet_strength, fenwick_max_wet_strength[index]
        )
        index -= index & -index

    if best_prior_wet_strength < wet_strength:
        pareto_indices.append(original_index)

    # Add this candidate to the prefix-maximum Fenwick tree.
    index = query_index
    while index <= level_count:
        fenwick_max_wet_strength[index] = max(
            fenwick_max_wet_strength[index], wet_strength
        )
        index += index & -index

pareto_candidates = candidate_predictions.iloc[pareto_indices].copy()
assert len(pareto_candidates) > 0
assert pareto_candidates["TEM"].between(5.0, 8.0).all()
assert pareto_candidates[chain_extender_cols].sum(axis=1).le(6.0).all()
assert all(col in pareto_candidates.columns for col in std_cols)

# These frontier members show the single-objective extremes and trade-offs.
pareto_extremes = pd.concat(
    [
        pareto_candidates.loc[[pareto_candidates[density_col].idxmin()]],
        pareto_candidates.loc[[pareto_candidates[dry_strength_col].idxmax()]],
        pareto_candidates.loc[[pareto_candidates[wet_strength_col].idxmax()]],
    ]
).drop_duplicates()

print(f"Found {len(pareto_candidates):,} Pareto-optimal formulas out of {len(candidate_predictions):,} candidates.")
print("Pareto-front extremes (lowest density, highest dry strength, highest wet strength):")
display(pareto_extremes[feature_cols + objective_cols + std_cols])
print("\nFirst 20 Pareto candidates, ordered by predicted density:")
display(
    pareto_candidates.sort_values(density_col)[feature_cols + objective_cols + std_cols].head(20)
)

In [ ]:
# Load measured targets and standard errors for the observed-formula plot.
from pathlib import Path
import pandas as pd
import numpy as np

project_root = Path.cwd().resolve().parent if 'notebooks' in str(Path.cwd()) else Path.cwd().resolve()
excel_path = project_root / 'data' / 'raw' / 'Compiled Data.xlsx'
df = pd.read_excel(excel_path)

formula_cols = ["TEM", "Joncryl", "NC513", "PEG-DE", "NGDE"]
measured_target_cols = [
    "Avg_Density_g_cm3",
    "Dry Strength Mean",
    "Wet Strength Mean",
]
measured_se_cols = [
    "Avg_Density SE",
    "Dry Strength SE",
    "Wet Strength SE",
]

measured_cols = [
    col
    for target, se in zip(measured_target_cols, measured_se_cols)
    for col in (target, se)
]
df = df.loc[:, formula_cols + measured_cols]

df["std_Avg_Density_g_cm3"] = df["Avg_Density SE"] * np.sqrt(3)
df["std_Dry Strength Mean"] = df["Dry Strength SE"] * np.sqrt(3)
df["std_Wet Strength Mean"] = df["Wet Strength SE"] * np.sqrt(3)

assert len(df) == 25
assert df[measured_target_cols + measured_se_cols].notna().all().all()
print(df.head())

In [ ]:
# Plot measured means +/- observed SD and Pareto predictions +/- model SD.
import matplotlib.pyplot as plt
import numpy as np

target_std_pairs = {
    "Avg_Density_g_cm3": "std_Avg_Density_g_cm3",
    "Dry Strength Mean": "std_Dry Strength Mean",
    "Wet Strength Mean": "std_Wet Strength Mean",
}

target_labels = {
    "Avg_Density_g_cm3": "Average density (g/cm^3)",
    "Dry Strength Mean": "Dry strength",
    "Wet Strength Mean": "Wet strength",
}

pareto_top5 = pareto_candidates.sort_values(density_col).head(20)
pareto_x = len(df) + np.arange(1, len(pareto_top5) + 1)
prediction_cols = [f"predicted_{target}" for target in target_cols]
model_std_cols = [f"std_{target}" for target in target_cols]
pareto_plot_table = pareto_top5[feature_cols + prediction_cols + model_std_cols].copy()
pareto_plot_table.insert(0, "Candidate row", pareto_top5.index)
pareto_plot_table.insert(0, "Pareto rank", np.arange(1, len(pareto_top5) + 1))
pareto_plot_display = pareto_plot_table.reset_index(drop=True).copy()
pareto_plot_display[feature_cols] = pareto_plot_display[feature_cols].round(1)
print("Formulas plotted (ranked by predicted density):")
display(pareto_plot_display)

ingredient_cols = ["Joncryl", "NC513", "PEG-DE", "NGDE"]
ingredient_presence = df[ingredient_cols].notna() & df[ingredient_cols].ne(0)
ingredient_count = ingredient_presence.sum(axis=1)

df["formula_group"] = "TEM only"
df.loc[ingredient_count > 1, "formula_group"] = "Multiple"

single_ingredient = ingredient_count == 1
df.loc[single_ingredient, "formula_group"] = (
    ingredient_presence.loc[single_ingredient].idxmax(axis=1)
)

formula_groups = [
    group
    for group in ingredient_cols + ["Multiple", "TEM only"]
    if group in df["formula_group"].values
]
formula_colors = dict(zip(formula_groups, plt.get_cmap("tab10").colors))

fig, axes = plt.subplots(1, len(target_std_pairs), figsize=(18, 5), sharex=True)
measured_ticks = np.unique(np.append(np.arange(1, len(df) + 1, 5), len(df)))
pareto_tick_ranks = np.arange(1, len(pareto_top5) + 1)
pareto_ticks = len(df) + pareto_tick_ranks

for ax, target in zip(axes, target_std_pairs):
    observed_std_col = target_std_pairs[target]
    plot_data = df[[target, observed_std_col, "formula_group"]].dropna(
        subset=[target, observed_std_col]
    )

    for group in formula_groups:
        group_data = plot_data.loc[plot_data["formula_group"] == group]
        means = group_data[target].to_numpy()
        standard_deviations = group_data[observed_std_col].to_numpy()
        formulation_indices = group_data.index.to_numpy() + 1

        ax.errorbar(
            formulation_indices,
            means,
            yerr=standard_deviations,
            fmt="o",
            capsize=3,
            alpha=0.8,
            color=formula_colors[group],
            label=group,
        )

    prediction_col = f"predicted_{target}"
    predictive_std_col = f"std_{target}"
    ax.errorbar(
        pareto_x,
        pareto_top5[prediction_col],
        yerr=pareto_top5[predictive_std_col],
        fmt="s",
        markersize=5,
        capsize=3,
        color="black",
        ecolor="black",
        label="Top 5 Pareto (model mean +/- SD)",
    )

    ax.axvline(len(df) + 0.5, color="gray", linestyle="--", alpha=0.7)
    ax.set_title(target_labels[target])
    ax.set_xlabel("Measured formula index; Pareto ranks follow")
    ax.set_ylabel("Measured / predicted value")
    ax.set_xlim(0, len(df) + len(pareto_top5) + 1)
    ax.set_xticks(
        np.concatenate([measured_ticks, pareto_ticks]),
        [str(index) for index in measured_ticks]
        + [f"P{rank}" for rank in pareto_tick_ranks],
    )
    ax.grid(axis="y", alpha=0.25)
    ax.legend(fontsize=8)

fig.tight_layout()
plt.show()